In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch import nn
import torch
from IPython.display import Markdown
import os

In [2]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Category or class names

In [3]:
# Map labels to integers
categories=['負面','正面']

In [4]:

label_to_id = { cate : i for i, cate in enumerate(categories)}

In [5]:
label_to_id

{'負面': 0, '正面': 1}

In [6]:
id_to_label = { i : cate for i, cate in enumerate(categories)}

In [7]:
id_to_label

{0: '負面', 1: '正面'}

In [13]:
import torch
import os
import torch.nn.functional as F
from torch import nn

class QwenForClassifier(nn.Module):
    def __init__(self, base_model, hidden_size, num_labels):
        super(QwenForClassifier, self).__init__()
        # 凍結 base model 的參數
        self.base_model = base_model
        
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # 注意力池化機制
        self.attention_pooler = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # 多層融合權重 (最後4層)
        self.layer_weights = nn.Parameter(torch.ones(4) / 4)
        
        # 增強型分類器
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, num_labels)
        )
        
        # 保存配置
        self.config = base_model.config
        self.config.num_labels = num_labels
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 獲取所有隱藏層狀態
        outputs = self.base_model(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        # 獲取最後4層隱藏狀態
        hidden_states = outputs.hidden_states
        if hidden_states is None:
            # 如果模型沒有返回hidden_states，使用last_hidden_state
            last_hidden = outputs.last_hidden_state
            sequence_output = last_hidden
        else:
            # 融合最後4層 (或可用層數)
            last_layers = hidden_states[-4:] if len(hidden_states) >= 4 else hidden_states[1:]
            layer_weights = F.softmax(self.layer_weights[:len(last_layers)], dim=0)
            
            # 加權融合多層特徵
            sequence_output = torch.zeros_like(last_layers[0])
            for i, layer in enumerate(last_layers):
                sequence_output += layer_weights[i].unsqueeze(-1).unsqueeze(-1) * layer
        
        # 注意力池化
        attention_scores = self.attention_pooler(sequence_output)
        attention_probs = F.softmax(attention_scores, dim=1)
        context_vector = torch.matmul(attention_probs.transpose(-1, -2), sequence_output).squeeze(1)
        
        # 也計算平均池化向量
        mean_pooled = torch.mean(sequence_output, dim=1)
        
        # 結合注意力池化和平均池化 (殘差連接)
        combined_repr = context_vector + mean_pooled
            
        # 分類預測
        logits = self.classifier(combined_repr)
        
        # 計算損失
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            
        return {"loss": loss, "logits": logits}
    
    def save_model(self, output_dir=None):
        """保存分類器權重和配置"""
        os.makedirs(output_dir, exist_ok=True)
        
        # 保存分類器權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.classifier.state_dict(),
            'attention_pooler': self.attention_pooler.state_dict(),
            'layer_weights': self.layer_weights,
            'config': {
                'num_labels': self.config.num_labels,
                'hidden_size': self.config.hidden_size
            }
        }
        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")
    
    def load_model(self, model_dir, device=None):
        """載入分類器權重"""
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            
        classifier_path = os.path.join(model_dir, "classifier_weights.pt")
        if os.path.exists(classifier_path):
            model_dict = torch.load(classifier_path, map_location=device, weights_only=True)
            
            # 載入各組件
            self.classifier.load_state_dict(model_dict['classifier'])
            self.attention_pooler.load_state_dict(model_dict['attention_pooler'])
            self.layer_weights.data = model_dict['layer_weights'].to(device)
            
            print(f"已載入分類器權重: {classifier_path}")
            return True
        else:
            print(f"警告: 找不到分類器權重檔案 {classifier_path}")
            return False

# Load

In [9]:
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [11]:
# 載入基礎模型
full_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

In [14]:

hidden_size = full_model.config.hidden_size

# 建立分類模型
model = QwenForClassifier(full_model.model, hidden_size, num_labels=2)


#
model_path = "trained_classifier_v4-5epochs-acc0.93"
# model_path = "checkpoints_v3\checkpoint-4145"
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

model.load_model(model_path, device=device)


# 載入分類器權重
# classifier_path = "trained_classifier_v4"
# classifier_weights_path = os.path.join(classifier_path, "classifier_weights.pt")
# if os.path.isfile(classifier_weights_path):
#     classifier_weights = torch.load(classifier_weights_path, map_location=device, weights_only=True)
#     model.classifier.load_state_dict(classifier_weights)
# else:
#     print(f"Warning: 在 {classifier_weights_path} 找不到分類器權重")
    
# 移動到指定設備
model = model.to(device)

已載入分類器權重: trained_classifier_v4-5epochs-acc0.93\classifier_weights.pt


In [15]:
full_model


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [16]:
model

QwenForClassifier(
  (base_model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMS

In [17]:
model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

# 分類模型怎麼用?

In [18]:
import numpy as np
from transformers import AutoTokenizer

# Function to make predictions
def predict_sentiment(text, model, tokenizer, device):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    logits = outputs["logits"]
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # Get the class name using id_to_label
    predicted_label = id_to_label[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "sentiment": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_label[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }


In [19]:
text = "今天天氣真好，我很開心"
predict_sentiment(text, model, tokenizer, device)

{'text': '今天天氣真好，我很開心',
 'sentiment': '正面',
 'confidence': 1.0,
 'probabilities': {'負面': 0.0, '正面': 1.0}}

In [20]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '負面',
 'confidence': 1.0,
 'probabilities': {'負面': 1.0, '正面': 0.0}}

In [21]:
text = "這家餐廳的食物美味，環境也很舒適"
predict_sentiment(text, model, tokenizer, device)

{'text': '這家餐廳的食物美味，環境也很舒適',
 'sentiment': '正面',
 'confidence': 1.0,
 'probabilities': {'負面': 0.0, '正面': 1.0}}

In [22]:
text = "我沒有很討厭這部電影"
predict_sentiment(text, model, tokenizer, device)

{'text': '我沒有很討厭這部電影',
 'sentiment': '負面',
 'confidence': 0.98,
 'probabilities': {'負面': 0.98, '正面': 0.02}}

In [23]:
text = "我沒有很討厭這部電影，但也不會推薦給朋友"
predict_sentiment(text, model, tokenizer, device)

{'text': '我沒有很討厭這部電影，但也不會推薦給朋友',
 'sentiment': '負面',
 'confidence': 1.0,
 'probabilities': {'負面': 1.0, '正面': 0.0}}

# 原始語言模型怎麼用?

In [24]:

#   generated_ids = model.generate(**model_inputs, 
#                                  max_new_tokens=512, 
#                                  do_sample=True, 
#                                  pad_token_id=tokenizer.eos_token_id)

def generate_text(input_prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response


In [25]:
text="給出三個保持健康的提示。"
result = generate_text(text)
Markdown(result)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1. 健康的生活方式，包括均衡的饮食、定期的运动和充足的睡眠。
2. 保持积极的心态，学会应对压力和挑战，并培养自我关爱的习惯。
3. 定期进行体检，及时发现并解决潜在的健康问题。

In [26]:
%%time
text="我們如何減少空氣污染？請給幾項重要的建議。"
result = generate_text(text)
Markdown(result)

CPU times: total: 4.36 s
Wall time: 4.81 s


要減少空氣污染，可以從以下幾個方面入手：

1. 降低車輛排放：選擇更高效的車型、使用電動或混合动力車、節油的技術和設備。

2. 提高能源效率：對家庭和工業設備進行優化，提高能源效率。

3. 增加公共交通系統：建立更多公共交通工具和車道，減少私家車出行。

4. 調整建築設計：建造低噪音建筑，減少噪音對環境的影響。

5. 使用可再生能源：利用風能、太阳能等可再生能源來供應電力。

6. 遵循環保政策：遵守國家和地方的環保規定，如限行汽車排放標準。

7. 加強教育和宣傳：提高公众對環保意識，讓更多的人了解如何減少污染源。

8. 支持環保企業和產品：支持那些生產環保產品的公司，以促進環保產業的发展。

9. 對於污染嚴重的城市和地區，政府應該採取措施，例如限制車輛排放、禁止非必要燃燒化石燃料等。

10. 推廣低碳生活方式：比如節約用水、減少食物浪费等。

這些措施都需要大家共同努力，才能有效減少空氣污染。

In [27]:
%%time
text="亞洲最高的山?"
result = generate_text(text)
Markdown(result)

CPU times: total: 312 ms
Wall time: 336 ms


珠穆朗玛峰是亚洲最高的山峰，海拔8,848米。

In [28]:
full_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((